# 01 — Tổng quan và kiểm tra dữ liệu Valorant

Mục tiêu của notebook:

1. Kiểm tra quy mô, phạm vi thời gian và mức độ thiếu dữ liệu.
2. Hiểu vì sao KAST bị loại khỏi mô hình.
3. Chạy quy trình làm sạch và xem số ván/người chơi còn lại.

Dữ liệu gốc: `data/raw/valorant.sqlite`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from valorant_mining.config import DB_PATH, OUTPUT_TABLES
from valorant_mining.db import read_sql, table_counts

print("DB:", DB_PATH, DB_PATH.exists())
table_counts()


## 1. Quy mô dữ liệu và phân bố bản đồ

In [ ]:
maps = read_sql("SELECT Map, COUNT(*) AS games FROM Games GROUP BY Map ORDER BY games DESC")
dates = read_sql("SELECT MIN(Date) AS first_date, MAX(Date) AS last_date FROM Matches")
display(dates)
display(maps)


## 2. Thiếu dữ liệu

KAST gần như thiếu toàn bộ, nên không thể dùng làm biến trung tâm. Không thay giá trị thiếu bằng 0 vì sẽ tạo ra tín hiệu giả.


In [ ]:
missing = read_sql('''
SELECT
  SUM(PlayerID IS NULL) AS player_id_null,
  SUM(Agent IS NULL OR TRIM(Agent) = '') AS agent_blank,
  SUM(ACS IS NULL) AS acs_null,
  SUM(KAST_Percent IS NULL) AS kast_null,
  SUM(Num_2Ks IS NULL) AS multikill_null,
  COUNT(*) AS rows
FROM Game_Scoreboard
''')
missing


## 3. Làm sạch và kiểm tra đội hình

Quy tắc: mỗi đội đúng 5 người chơi, 5 Agent khác nhau; mỗi ván đúng 2 đội; số round lấy từ RoundHistory nếu hợp lệ.


In [ ]:
from valorant_mining.cleaning import build_clean_data

clean = build_clean_data()
pd.Series(clean.audit).to_frame("value")


In [ ]:
comps = clean.compositions
summary = (comps.groupby("Map")
           .agg(teams=("GameID", "size"), games=("GameID", "nunique"), labeled=("won", "count"))
           .sort_values("teams", ascending=False))
summary


## 4. Nhận xét

- Chỉ khoảng 88% ván tạo được đội hình 2 × 5 người và 5 Agent.
- KAST thiếu gần như toàn bộ → loại.
- Breeze/Fracture có ít mẫu; luật của hai map này cần thận trọng.
- Bước tiếp theo: `02_phan_cum_nguoi_choi.ipynb`.
